<a href="https://colab.research.google.com/github/Maddox159-crypto/ESAA_assignment/blob/main/0904_%EC%84%B8%EC%85%98_%EB%AA%A8%EB%8D%B8%ED%9B%88%EB%A0%A8_%EC%97%B0%EC%8A%B5%EB%AC%B8%EC%A0%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **모델 훈련 연습 문제**
___
- 출처 : 핸즈온 머신러닝 Ch04 연습문제 1, 5, 9, 10
- 개념 문제의 경우 텍스트 셀을 추가하여 정답을 적어주세요.

### **1. 수백만 개의 특성을 가진 훈련 세트에서는 어떤 선형 회귀 알고리즘을 사용할 수 있을까요?**
___


확률적 경사 하강법이나 미니배치 경사 하강법을 사용할 수 있다

훈련 세트가 메모리 크기에 맞으면 배치 경사 하강법도 가능하다.

### **2. 배치 경사 하강법을 사용하고 에포크마다 검증 오차를 그래프로 나타내봤습니다. 검증 오차가 일정하게 상승되고 있다면 어떤 일이 일어나고 있는 걸까요? 이 문제를 어떻게 해결할 수 있나요?**
___

검증 오차가 일정하게 상승되고 있다면 학습률이 너무 높아서 알고리즘이 발산하고 있다는 뜻이기 때문에 학습률을 낮춰서 다시 훈련시켜야 함.  


### **3. 릿지 회귀를 사용했을 때 훈련 오차가 검증 오차가 거의 비슷하고 둘 다 높았습니다. 이 모델에는 높은 편향이 문제인가요, 아니면 높은 분산이 문제인가요? 규제 하이퍼파라미터 $\alpha$를 증가시켜야 할까요 아니면 줄여야 할까요?**
___

모델의 편향이 높은 상태임. 모델이 데이터의 패턴을 충분히 학습하지 못했음을 의미함. 따라서 알파를 내려야 하고 이를 통해 규제를 완화해서 자유도를 높여 편향을 줄여야 함

### **4. 다음과 같이 사용해야 하는 이유는?**
___
- 평범한 선형 회귀(즉, 아무런 규제가 없는 모델) 대신 릿지 회귀
- 릿지 회귀 대신 라쏘 회귀
- 라쏘 회귀 대신 엘라스틱넷

규제가 없는 선형 회귀보다 약간의 규제가 있는 모델이 일반적으로 더 좋은 성능을 가짐. Ridge 회귀는 가중치의 크기를 제한해서 분산을 줄이고 과대적합을 방지해 줌.   
Lasso 회귀는 L1 규제를 사용하여 중요하지 않은 특성의 가중치를 0으로 들기에 즉, 자동으로 feature selection을 수행하여 sparse model을 생성하므로, 실제로 유용한 특성이 몇 개뿐이라고 의심될 때 유리함.  
Lasso는 특성 수가 샘플 수보다 많거나, 몇 개의 특성이 강한 상관관계를 가질 때 불안정하게 작동하거나 일부 특성을 무작위로 선택하는 경향이 있음.
ElasticNet은 L1과 L2 규제를 혼합 사용하여 이러한 Lasso의 한계를 보완하고 안정적인 성능을 제공함.

### **추가) 조기 종료를 사용한 배치 경사 하강법으로 iris 데이터를 활용해 소프트맥스 회귀를 구현해보세요(사이킷런은 사용하지 마세요)**


---



In [1]:
import numpy as np

from sklearn.datasets import load_iris
iris = load_iris()
X = iris["data"]
y = iris["target"]

X_with_bias = np.c_[np.ones([len(X), 1]), X]

np.random.seed(42)
total_size = len(X_with_bias)
test_ratio = 0.2
val_ratio = 0.2

test_size = int(total_size * test_ratio)
val_size = int(total_size * val_ratio)
train_size = total_size - test_size - val_size

rnd_indices = np.random.permutation(total_size)

X_train = X_with_bias[rnd_indices[:train_size]]
y_train = y[rnd_indices[:train_size]]
X_valid = X_with_bias[rnd_indices[train_size:-test_size]]
y_valid = y[rnd_indices[train_size:-test_size]]
X_test = X_with_bias[rnd_indices[-test_size:]]
y_test = y[rnd_indices[-test_size:]]

def to_one_hot(y):
    n_classes = y.max() + 1
    m = len(y)
    Y_one_hot = np.zeros((m, n_classes))
    Y_one_hot[np.arange(m), y] = 1
    return Y_one_hot

Y_train_one_hot = to_one_hot(y_train)
Y_valid_one_hot = to_one_hot(y_valid)
Y_test_one_hot = to_one_hot(y_test)

mean = X_train[:, 1:].mean(axis=0)
std = X_train[:, 1:].std(axis=0)

X_train[:, 1:] = (X_train[:, 1:] - mean) / std
X_valid[:, 1:] = (X_valid[:, 1:] - mean) / std
X_test[:, 1:] = (X_test[:, 1:] - mean) / std

In [2]:
def softmax(logits):
    exps = np.exp(logits - np.max(logits, axis=1, keepdims=True)) # 수치 안정성을 위해 np.max 감산
    return exps / np.sum(exps, axis=1, keepdims=True)

In [3]:
n_inputs = X_train.shape[1]
n_outputs = len(np.unique(y))

eta = 0.1
n_iterations = 5001
m = len(X_train)
epsilon = 1e-7

Theta = np.random.randn(n_inputs, n_outputs)

best_loss = np.inf
patience = 200
patience_counter = 0
best_Theta = None

for iteration in range(n_iterations):
    logits = X_train.dot(Theta)
    Y_proba = softmax(logits)

    error = Y_proba - Y_train_one_hot
    gradients = (1 / m) * X_train.T.dot(error)

    Theta = Theta - eta * gradients

    logits_valid = X_valid.dot(Theta)
    Y_proba_valid = softmax(logits_valid)
    valid_loss = -np.mean(np.sum(Y_valid_one_hot * np.log(Y_proba_valid + epsilon), axis=1))

    if valid_loss < best_loss:
        best_loss = valid_loss
        best_Theta = Theta.copy()
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early Stopping! Iteration: {iteration}, Best Validation Loss: {best_loss:.4f}")
            break

Theta = best_Theta

In [4]:
logits_test = X_test.dot(Theta)
Y_proba_test = softmax(logits_test)
y_predict = np.argmax(Y_proba_test, axis=1)

accuracy = np.mean(y_predict == y_test)
print(f"Test Accuracy: {accuracy * 100:.2f}%")

Test Accuracy: 100.00%
